In [1]:
%pip install -qU pypdf langchain-community langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_community.document_loaders import PyPDFLoader

pdf_file_path = "./income_tax.pdf"

loader = PyPDFLoader(file_path=pdf_file_path)
pages = []

async for page in loader.alazy_load():
    pages.append(page)

c:\Users\JJH\anaconda3\envs\langchain\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
pages[35]

Document(metadata={'producer': 'iText 2.1.7 by 1T3XT', 'creator': 'PyPDF', 'creationdate': '2026-04-21T20:49:56+09:00', 'moddate': '2026-04-21T20:49:56+09:00', 'source': './income_tax.pdf', 'total_pages': 137, 'page': 35, 'page_label': '36'}, page_content='법제처                                                            36                                                       국가법령정보센터\n소득세법\n② 다음 각 호의 어느 하나에 해당하는 경우에는 제1항에 따른 공제[이하 “기장세액공제”(記帳稅額控除)라 한다]를\n적용하지 아니한다.\n1. 비치ㆍ기록한 장부에 의하여 신고하여야 할 소득금액의 100분의 20 이상을 누락하여 신고한 경우\n2. 기장세액공제와 관련된 장부 및 증명서류를 해당 과세표준확정신고기간 종료일부터 5년간 보관하지 아니한 경\n우. 다만, 천재지변 등 대통령령으로 정하는 부득이한 사유에 해당하는 경우에는 그러하지 아니하다.\n③ 기장세액공제에 관하여 필요한 사항은 대통령령으로 정한다.\n[전문개정 2009. 12. 31.]\n \n제56조의3(전자계산서 발급 전송에 대한 세액공제) ① 총수입금액 등을 고려하여 대통령령으로 정하는 사업자가 제\n163조제1항 후단에 따른 전자계산서를 2027년 12월 31일까지 발급(제163조제8항에 따라 전자계산서 발급명세를\n국세청장에게 전송하는 경우로 한정한다)하는 경우에는 전자계산서 발급 건수 등을 고려하여 대통령령으로 정하는\n금액을 해당 과세기간의 사업소득에 대한 종합소득산출세액에서 공제할 수 있다. 이 경우 공제한도는 연간 100만원\n으로 한다. <개정 2021. 12. 8., 2024. 12. 31.

In [4]:
%pip install -q py-zerox # zerox 라이브러리 파싱용

Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


In [5]:
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
%pip install -q nest_asyncio

Note: you may need to restart the kernel to use updated packages.


In [8]:
import nest_asyncio
nest_asyncio.apply()

In [9]:
from pyzerox import zerox
import os
import json
import asyncio

### Model Setup (Use only Vision Models) Refer: https://docs.litellm.ai/docs/providers ###

model = "gpt-4o-mini"
os.environ["OPENAI_API_KEY"]

# Keep pyzerox effectively sequential to avoid TPM spikes on large PDFs.
CONCURRENCY = 1
BATCH_SIZE = 5
MAX_RETRIES = 6
BASE_DELAY_SECONDS = 3

kwargs = {
    "concurrency": CONCURRENCY,
}

custom_system_prompt = None

def chunked_pages(total_pages: int, batch_size: int):
    for start in range(1, total_pages + 1, batch_size):
        end = min(start + batch_size - 1, total_pages)
        yield list(range(start, end + 1))

async def run_zerox_batch(file_path: str, output_dir: str, page_batch: list[int]):
    for attempt in range(MAX_RETRIES):
        try:
            return await zerox(
                file_path=file_path,
                model=model,
                output_dir=output_dir,
                custom_system_prompt=custom_system_prompt,
                select_pages=page_batch,
                **kwargs,
            )
        except Exception as exc:
            message = str(exc)
            is_rate_limit = "RateLimitError" in message or "rate limit" in message.lower()
            if attempt == MAX_RETRIES - 1 or not is_rate_limit:
                raise

            delay = BASE_DELAY_SECONDS * (2 ** attempt)
            print(f"Rate limit on pages {page_batch}. retry={attempt + 1}/{MAX_RETRIES} sleep={delay}s")
            await asyncio.sleep(delay)

async def main():
    file_path = "./income_tax.pdf"
    output_dir = "./document"
    total_pages = 137

    batch_results = []
    for page_batch in chunked_pages(total_pages, BATCH_SIZE):
        print(f"Processing pages {page_batch[0]}-{page_batch[-1]}")
        batch_result = await run_zerox_batch(file_path, output_dir, page_batch)
        batch_results.append(batch_result)
        await asyncio.sleep(1)

    return batch_results

results = asyncio.run(main())
print(results[-1])

ERROR:root:Error converting PDF to images: Unable to get page count. Is poppler installed and in PATH?


TypeError: 'NoneType' object is not iterable